## CELL 1 — Setup

In [ ]:
!pip install -q albumentations onnx onnxruntime onnxscript

import os
import sys
import time
import json
import random
import gc
import warnings
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')

# ── Environment & Seeds ────────────────────────────────────
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything()

# ── Hardware ───────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

# ── Configuration ──────────────────────────────────────────
BATCH_SIZE   = 4
ACCUM_STEPS  = 8     # Effective batch = 32
EPOCHS       = 70
WARMUP_EP    = 5
PATIENCE     = 25
LR           = 2e-4
MIN_LR       = 1e-7
WEIGHT_DECAY = 2e-4

KAGGLE_WORKING = '/kaggle/working'
CHECKPOINT_DIR = f'{KAGGLE_WORKING}/checkpoints'
EXPORT_DIR     = f'{KAGGLE_WORKING}/export'
BEST_CKPT      = f'{CHECKPOINT_DIR}/best_model.pth'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)


## CELL 2 — Dataset Discovery

In [ ]:
# ── Find Dataset in /kaggle/input/ ─────────────────────────
import os

# Manually set correct paths for P3M 10K dataset
BASE = '/kaggle/input/datasets/rahulbhalley/p3m-10k/P3M-10k'

# Try training set first
TRAIN_IMG  = f'{BASE}/train/blurred_image'
TRAIN_MASK = f'{BASE}/train/mask'

# Fallback paths
ALT_IMG  = f'{BASE}/validation/P3M-500-P/blurred_image'
ALT_MASK = f'{BASE}/validation/P3M-500-P/mask'

if os.path.exists(TRAIN_IMG) and os.path.exists(TRAIN_MASK):
    IMAGE_DIR = TRAIN_IMG
    MASK_DIR  = TRAIN_MASK
    print(f'Found TRAINING set!')
elif os.path.exists(ALT_IMG):
    IMAGE_DIR = ALT_IMG
    MASK_DIR  = ALT_MASK
    print(f'Using validation set (training set not found)')
else:
    # Walk and show everything
    print('Walking /kaggle/input to find dataset...')
    for root, dirs, files in os.walk('/kaggle/input'):
        imgs = [f for f in files if f.endswith(
            ('.jpg','.jpeg','.png'))]
        if len(imgs) > 100:
            print(f'{root}: {len(imgs)} images')

print(f'\nImages : {IMAGE_DIR}')
print(f'Masks  : {MASK_DIR}')
print(f'Image count: {len(os.listdir(IMAGE_DIR))}')
print(f'Mask count:  {len(os.listdir(MASK_DIR))}')

## CELL 3 — Dataset + Loaders

In [ ]:
# ── Augmentations ──────────────────────────────────────────
def make_transforms(size, training=True):
    if training:
        return A.Compose([
            A.Resize(size, size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.1),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.25, rotate_limit=20, p=0.7),
            A.ElasticTransform(alpha=120, sigma=6, alpha_affine=6, p=0.3),
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),
            A.GaussianBlur(blur_limit=(3, 7), p=0.3),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
            A.CoarseDropout(max_holes=6, max_height=48, max_width=48, p=0.3),
            A.RandomShadow(p=0.2),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(size, size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

# ── Dataset ────────────────────────────────────────────────
class SegDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, msk_path = self.pairs[idx]
        try:
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            mask = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
            mask = (mask > 127).astype(np.float32)

            if self.transform:
                aug = self.transform(image=image, mask=mask)
                image = aug['image']
                mask  = aug['mask'].unsqueeze(0)
                
            return {'image': image, 'mask': mask}
        except Exception as e:
            sz = 192 
            for t in self.transform.transforms:
                if isinstance(t, A.Resize):
                    sz = t.height
                    break
            return {'image': torch.zeros(3, sz, sz), 'mask': torch.zeros(1, sz, sz)}

def build_pairs(img_dir, msk_dir):
    if not img_dir or not msk_dir: return []
    IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp'}
    img_files = {Path(f).stem: os.path.join(img_dir, f) for f in os.listdir(img_dir) if Path(f).suffix.lower() in IMG_EXT}
    msk_files = {Path(f).stem: os.path.join(msk_dir, f) for f in os.listdir(msk_dir) if Path(f).suffix.lower() in IMG_EXT}
    common = sorted(set(img_files) & set(msk_files))
    pairs = [(img_files[s], msk_files[s]) for s in common]
    print(f'Matched pairs: {len(pairs)}')
    return pairs

all_pairs = build_pairs(IMAGE_DIR, MASK_DIR)
random.Random(42).shuffle(all_pairs)

n = len(all_pairs)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_pairs = all_pairs[:n_train]
val_pairs   = all_pairs[n_train:n_train + n_val]
test_pairs  = all_pairs[n_train + n_val:]

def build_loaders(size):
    print(f'Building loaders for size: {size}x{size}')
    train_tf = make_transforms(size, training=True)
    val_tf   = make_transforms(size, training=False)

    train_ds = SegDataset(train_pairs, train_tf)
    val_ds   = SegDataset(val_pairs, val_tf)
    test_ds  = SegDataset(test_pairs, val_tf)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, pin_memory=False, drop_last=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=0, pin_memory=False
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=0, pin_memory=False
    )
    return train_loader, val_loader, test_loader

if all_pairs:
    train_loader, val_loader, test_loader = build_loaders(192)


## CELL 4 — CBAM

In [ ]:
# ── CBAM ───────────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.mlp = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        gap = x.mean(dim=(2, 3))           
        gmp = x.amax(dim=(2, 3))           
        gate = torch.sigmoid(self.mlp(gap) + self.mlp(gmp)) 
        return x * gate.view(B, C, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)  
        mx  = x.amax(dim=1, keepdim=True)  
        gate = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * gate

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention()

    def forward(self, x):
        return self.sa(self.ca(x))

print('CBAM defined.')


## CELL 5 — Encoder Blocks

In [ ]:
# ── MobileNetV3 Components ─────────────────────────────────
def hswish(x):
    return x * F.relu6(x + 3, inplace=True) / 6

def hsigmoid(x):
    return F.relu6(x + 3, inplace=True) / 6

class HSwish(nn.Module):
    def forward(self, x): return hswish(x)

class SqueezeExcite(nn.Module):
    def __init__(self, ch, reduction=4):
        super().__init__()
        mid = max(ch // reduction, 4)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, ch, bias=False),
        )

    def forward(self, x):
        s = torch.sigmoid(self.fc(x))
        return x * s.view(x.size(0), x.size(1), 1, 1)

class InvertedResidual(nn.Module):
    def __init__(self, in_ch, out_ch, stride, expand_ratio, kernel=3, use_se=False):
        super().__init__()
        self.use_skip = (stride == 1 and in_ch == out_ch)
        mid = int(in_ch * expand_ratio)

        layers = []
        if expand_ratio != 1:
            layers += [
                nn.Conv2d(in_ch, mid, 1, bias=False),
                nn.BatchNorm2d(mid),
                HSwish()
            ]
        layers += [
            nn.Conv2d(mid, mid, kernel, stride=stride, padding=kernel // 2, groups=mid, bias=False),
            nn.BatchNorm2d(mid),
            HSwish()
        ]
        if use_se:
            layers.append(SqueezeExcite(mid))
        layers += [
            nn.Conv2d(mid, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch)
        ]
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        out = self.block(x)
        if self.use_skip:
            out = out + x
        return out

print('Encoder blocks defined.')


## CELL 6 — Full Model

In [ ]:
# ── Decoder Block & Model ──────────────────────────────────
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU()
        )
        self.cbam = CBAM(out_ch)
        self.drop = nn.Dropout2d(0.1)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        x = self.cbam(x)
        x = self.drop(x)
        return x

class BoundaryRefinementHead(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + 1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 1, 1)
        )
    def forward(self, features, main_logit):
        x = torch.cat([features, main_logit], dim=1)
        return self.conv(x)

class BgRemovalNet(nn.Module):
    def __init__(self):
        super().__init__()
        # ── Encoder (MobileNetV3-Small) ──────────────
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            HSwish()
        ) 
        self.e1 = InvertedResidual(16, 16, stride=2, expand_ratio=1, kernel=3)           
        self.e2 = InvertedResidual(16, 24, stride=2, expand_ratio=4.5, kernel=3)         
        self.e3 = InvertedResidual(24, 40, stride=2, expand_ratio=4, kernel=5, use_se=True) 
        self.e4 = InvertedResidual(40, 96, stride=2, expand_ratio=6, kernel=5, use_se=True) 
        self.e5 = nn.Sequential(
            nn.Conv2d(96, 576, 1, bias=False),
            nn.BatchNorm2d(576),
            HSwish()
        ) 

        # ── Decoder (Enhanced) ───────────────────────
        # Respecting size mismatch fix: Mapping to correct progressive resolutions using skipped depths.
        self.d1 = DecoderBlock(576, 40, 192) # cat E5(576) + E3(40) -> 192 
        self.d2 = DecoderBlock(192, 24, 96)  # cat D1(192) + E2(24) -> 96
        self.d3 = DecoderBlock(96, 16, 48)   # cat D2(96) + E1(16) -> 48
        self.d4 = DecoderBlock(48, 16, 24)   # cat D3(48) + stem(16) -> 24
        
        self.d5 = nn.Sequential(
            nn.Conv2d(24, 8, 3, padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.GELU()
        ) 

        self.out_head = nn.Conv2d(8, 1, 1)
        self.boundary_head = BoundaryRefinementHead(8)

        # Deep Supervision Logits
        self.s2_head = nn.Conv2d(96, 1, 1)
        self.s3_head = nn.Conv2d(48, 1, 1)
        self.s4_head = nn.Conv2d(24, 1, 1)

    def forward(self, x):
        # ── Encode ─────────────────────
        s  = self.stem(x) 
        e1 = self.e1(s)   
        e2 = self.e2(e1)  
        e3 = self.e3(e2)  
        e4 = self.e4(e3)  
        e5 = self.e5(e4)  

        # ── Decode ─────────────────────
        d1 = self.d1(e5, e3) 
        d2 = self.d2(d1, e2) 
        d3 = self.d3(d2, e1) 
        d4 = self.d4(d3, s) 
        
        d5_feat = F.interpolate(d4, size=x.shape[2:], mode='bilinear', align_corners=False)
        d5_feat = self.d5(d5_feat)
        
        # ── Outputs ────────────────────
        main_logit = self.out_head(d5_feat)        
        boundary_logit = self.boundary_head(d5_feat, main_logit)

        if self.training:
            def up(t):
                if t.shape[2:] != x.shape[2:]:
                    return F.interpolate(t, size=x.shape[2:], mode='bilinear', align_corners=False)
                return t
            
            s2 = up(self.s2_head(d2))
            s3 = up(self.s3_head(d3))
            s4 = up(self.s4_head(d4))
            
            return {
                'main': main_logit,
                'boundary': boundary_logit,
                'deep': [s2, s3, s4]
            }

        return main_logit # Eval mode: RAW LOGITS

model = BgRemovalNet().to(device)
total = sum(p.numel() for p in model.parameters())
print(f'Model params: {total/1e6:.2f}M')


## CELL 7 — Loss Functions

In [ ]:
# ── Losses ─────────────────────────────────────────────────
class BCELoss(nn.Module):
    def forward(self, logits, target):
        return F.binary_cross_entropy_with_logits(logits, target, reduction='mean')

class DiceLoss(nn.Module):
    def forward(self, logits, target):
        pred  = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
        inter = (pred * target).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        return (1 - (2 * inter + 1) / (union + 1)).mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
        pt  = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

class IoULoss(nn.Module):
    def forward(self, logits, target):
        pred  = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
        inter = (pred * target).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) - inter
        return (1 - (inter + 1) / (union + 1)).mean()

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, logits, target):
        pred = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
        TP = (pred * target).sum(dim=(2, 3))
        FP = (pred * (1 - target)).sum(dim=(2, 3))
        FN = ((1 - pred) * target).sum(dim=(2, 3))
        tversky = (TP + 1) / (TP + self.alpha * FP + self.beta * FN + 1)
        return (1 - tversky).mean()

class BoundaryLoss(nn.Module):
    def forward(self, logits, target):
        dilated  = F.max_pool2d(target, 5, 1, 2)
        eroded   = -F.max_pool2d(-target, 5, 1, 2)
        boundary = (dilated - eroded).clamp(0, 1)
        weight   = 1.0 + 5.0 * boundary
        return F.binary_cross_entropy_with_logits(logits, target, weight=weight, reduction='mean')

class OHEMLoss(nn.Module):
    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
        bce = bce.view(-1)
        k = int(0.3 * bce.numel())
        if k > 0:
            hard_loss, _ = torch.topk(bce, k)
            return hard_loss.mean()
        return bce.mean()

class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce     = BCELoss()
        self.dice    = DiceLoss()
        self.focal   = FocalLoss()
        self.iou     = IoULoss()
        self.tversky = TverskyLoss()
        self.ohem    = OHEMLoss()
        self.bound   = BoundaryLoss()

    def forward(self, logits_dict, target):
        main_loss = (
            0.25 * self.bce(logits_dict['main'], target) +
            0.30 * self.dice(logits_dict['main'], target) +
            0.15 * self.focal(logits_dict['main'], target) +
            0.10 * self.iou(logits_dict['main'], target) +
            0.10 * self.tversky(logits_dict['main'], target) +
            0.10 * self.ohem(logits_dict['main'], target)
        )
        bound_loss = self.bound(logits_dict['boundary'], target)
        return main_loss + 0.3 * bound_loss

def get_pred(out):
    return out['main'] if isinstance(out, dict) else out

def deep_supervision_loss(preds, target, criterion):
    loss = criterion(preds, target)
    deep_outs = preds['deep']
    weights = [1.0, 0.4, 0.2, 0.1][:len(deep_outs)]
    for w, p in zip(weights, deep_outs):
        if p.shape[2:] != target.shape[2:]:
            p = F.interpolate(p, size=target.shape[2:], mode='bilinear', align_corners=False)
            
        bce = F.binary_cross_entropy_with_logits(p, target, reduction='mean')
        pred_sig = torch.sigmoid(p).clamp(1e-6, 1-1e-6)
        inter = (pred_sig * target).sum(dim=(2, 3))
        union = pred_sig.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice = (1 - (2 * inter + 1) / (union + 1)).mean()
        loss += w * (0.5 * bce + 0.5 * dice)
    return loss

criterion = CombinedLoss().to(device)
print('Loss functions defined.')


## CELL 8 — Metrics + Optimizer

In [ ]:
# ── Metrics + TTA + Optimizers ─────────────────────────────
def compute_iou(prob, target, thresh=0.5):
    pb    = (prob > thresh).float()
    tb    = (target > thresh).float()
    inter = (pb * tb).sum()
    union = pb.sum() + tb.sum() - inter
    return ((inter + 1e-7) / (union + 1e-7)).item()

def compute_dice(prob, target, thresh=0.5):
    pb    = (prob > thresh).float()
    tb    = (target > thresh).float()
    inter = (pb * tb).sum()
    return ((2 * inter + 1e-7) / (pb.sum() + tb.sum() + 1e-7)).item()

def compute_mae(prob, target):
    return (prob - target).abs().mean().item()

def tta_predict(model, img):
    model.eval()
    with torch.no_grad():
        p1 = torch.sigmoid(get_pred(model(img)))
        p2 = torch.sigmoid(get_pred(model(torch.flip(img, [3])))).flip([3])
        p3 = torch.sigmoid(get_pred(model(torch.flip(img, [2])))).flip([2])
    return (p1 + p2 + p3) / 3.0

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=MIN_LR
)

scaler = GradScaler()
print('Metrics and optimizers ready.')


## CELL 9 — Checkpoint System

In [ ]:
# ── Checkpoint System ──────────────────────────────────────
def save_checkpoint(epoch, best_iou, val_iou, is_best=False):
    ckpt = {
        'epoch'          : epoch,
        'model_state'    : model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state'   : scaler.state_dict(),
        'best_iou'       : best_iou,
        'val_iou'        : val_iou,
        'config': {
            'batch_size'  : BATCH_SIZE,
            'accum_steps' : ACCUM_STEPS,
            'lr'          : LR,
        }
    }
    torch.save(ckpt, BEST_CKPT)

    ep_path = f'{CHECKPOINT_DIR}/epoch_{epoch:03d}.pth'
    torch.save(ckpt, ep_path)

    ep_files = sorted(Path(CHECKPOINT_DIR).glob('epoch_*.pth'))
    for old in ep_files[:-2]:
        old.unlink(missing_ok=True)

    if is_best:
        torch.save(ckpt, f'{CHECKPOINT_DIR}/best_model.pth')
        print(f'  >> New best IoU: {best_iou:.4f}')

def load_checkpoint():
    if os.path.exists(BEST_CKPT):
        print(f'Found checkpoint — resuming...')
        ckpt = torch.load(BEST_CKPT, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        if 'scaler_state' in ckpt:
            scaler.load_state_dict(ckpt['scaler_state'])
        start_ep  = ckpt['epoch'] + 1
        best_iou  = ckpt['best_iou']
        print(f'Resumed from epoch {start_ep} | Best IoU: {best_iou:.4f}')
        return start_ep, best_iou
    print('No checkpoint found — starting fresh.')
    return 0, 0.0

start_epoch, best_iou = load_checkpoint()


## CELL 10 — Training Functions

In [ ]:
# ── Training & Validation ──────────────────────────────────
def train_one_epoch(epoch, train_loader):
    model.train()
    total_loss = 0.0
    n_batches = 0
    optimizer.zero_grad()

    for i, batch in enumerate(train_loader):
        imgs  = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)

        if random.random() < 0.2 and imgs.size(0) > 1:
            lam = np.random.beta(0.4, 0.4)
            idx = torch.randperm(imgs.size(0)).to(device)
            imgs = lam * imgs + (1 - lam) * imgs[idx]
            masks = lam * masks + (1 - lam) * masks[idx]

        masks = masks * 0.95 + 0.025

        with autocast():
            outs = model(imgs)
            loss = deep_supervision_loss(outs, masks, criterion) / ACCUM_STEPS

        if torch.isnan(loss) or torch.isinf(loss):
            print(f'  WARNING: NaN/Inf at step {i} — skip')
            optimizer.zero_grad()
            continue

        scaler.scale(loss).backward()

        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS
        n_batches += 1

    return total_loss / max(n_batches, 1)

def validate(val_loader, use_tta=False):
    model.eval()
    t_iou, t_dice, t_mae, n = 0, 0, 0, 0

    with torch.no_grad():
        for batch in val_loader:
            imgs  = batch['image'].to(device)
            masks = batch['mask'].to(device)

            if use_tta:
                prob = tta_predict(model, imgs)
            else:
                prob = torch.sigmoid(get_pred(model(imgs)))

            t_iou  += compute_iou(prob, masks)
            t_dice += compute_dice(prob, masks)
            t_mae  += compute_mae(prob, masks)
            n += 1

    return (t_iou / max(n, 1), t_dice / max(n, 1), t_mae / max(n, 1))

print('Training functions ready.')


## CELL 11 — Run Training

In [ ]:
# ── Training Loop ──────────────────────────────────────────
train_losses, val_ious = [], []
no_improve = 0

print(f'\n{"="*70}')
print(f' TRAINING START — {EPOCHS} epochs')
print(f' Effective batch: {BATCH_SIZE * ACCUM_STEPS}')
print(f' Starting epoch : {start_epoch}')
print(f'{"="*70}\n')

current_size = 320 if start_epoch >= 50 else 256 if start_epoch >= 20 else 192

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()

    target_size = 320 if epoch >= 50 else 256 if epoch >= 20 else 192
    if target_size != current_size:
        print(f"\n>>> PHASE TRANSITION: Adapting input size to {target_size}x{target_size} <<<")
        current_size = target_size
        train_loader, val_loader, test_loader = build_loaders(current_size)
        
        for pg in optimizer.param_groups:
            pg['lr'] = pg['lr'] * 0.5
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=10, T_mult=2, eta_min=MIN_LR
        )

    if epoch < WARMUP_EP:
        for pg in optimizer.param_groups:
            pg['lr'] = LR * (epoch + 1) / WARMUP_EP

    train_loss = train_one_epoch(epoch, train_loader)
    val_iou, val_dice, val_mae = validate(val_loader, use_tta=False)

    if epoch >= WARMUP_EP:
        scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    elapsed    = time.time() - t0
    remaining  = (EPOCHS - epoch - 1) * elapsed
    eta_h      = remaining / 3600

    print(f'Ep {epoch+1:3d}/{EPOCHS} | Loss: {train_loss:.4f} | IoU: {val_iou:.4f} | '
          f'Dice: {val_dice:.4f} | MAE: {val_mae:.4f} | LR: {current_lr:.1e} | '
          f'{elapsed:.0f}s | ETA: {eta_h:.1f}h')

    train_losses.append(train_loss)
    val_ious.append(val_iou)

    is_best = val_iou > best_iou
    if is_best:
        best_iou = val_iou
        no_improve = 0
    else:
        no_improve += 1

    save_checkpoint(epoch, best_iou, val_iou, is_best)

    if (epoch + 1) % 10 == 0:
        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        model.eval()
        sample = next(iter(val_loader))
        with torch.no_grad():
            imgs  = sample['image'][:4].to(device)
            masks = sample['mask'][:4]
            prob  = torch.sigmoid(get_pred(model(imgs)))
            pred  = (prob > 0.5).float().cpu()

        mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

        for j in range(min(4, imgs.size(0))):
            img_show = (imgs[j].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)
            axes[0,j].imshow(img_show)
            axes[0,j].set_title('Image')
            axes[0,j].axis('off')
            axes[1,j].imshow(pred[j,0], cmap='gray')
            axes[1,j].set_title(f'Pred IoU≈{val_iou:.3f}')
            axes[1,j].axis('off')

        plt.suptitle(f'Epoch {epoch+1}')
        plt.tight_layout()
        plt.savefig(f'{KAGGLE_WORKING}/preview_ep{epoch+1:03d}.png')
        plt.close(fig)
        model.train()

    torch.cuda.empty_cache()
    gc.collect()

    if no_improve >= PATIENCE:
        print(f'\nEarly stop: no improvement for {PATIENCE} epochs.')
        break

print(f'\n{"="*50}')
print(f'Training complete! Best Validation IoU: {best_iou:.4f}')
print(f'{"="*50}')


## CELL 12 — Export (Self-Contained)

In [ ]:
# ── Export & Validation (Self-Contained) ─────────────────
import os
import time
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnx
import onnxruntime as ort

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
KAGGLE_WORKING = '/kaggle/working'
CHECKPOINT_DIR = f'{KAGGLE_WORKING}/checkpoints'
EXPORT_DIR     = f'{KAGGLE_WORKING}/export'

def hswish(x): return x * F.relu6(x + 3, inplace=True) / 6
class HSwish(nn.Module):
    def forward(self, x): return hswish(x)

class SqueezeExcite(nn.Module):
    def __init__(self, ch, reduction=4):
        super().__init__()
        mid = max(ch // reduction, 4)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, mid, bias=False), nn.ReLU(inplace=True),
            nn.Linear(mid, ch, bias=False),
        )
    def forward(self, x):
        s = torch.sigmoid(self.fc(x))
        return x * s.view(x.size(0), x.size(1), 1, 1)

class InvertedResidual(nn.Module):
    def __init__(self, in_ch, out_ch, stride, expand_ratio, kernel=3, use_se=False):
        super().__init__()
        self.use_skip = (stride == 1 and in_ch == out_ch)
        mid = int(in_ch * expand_ratio)
        layers = []
        if expand_ratio != 1:
            layers += [nn.Conv2d(in_ch, mid, 1, bias=False), nn.BatchNorm2d(mid), HSwish()]
        layers += [
            nn.Conv2d(mid, mid, kernel, stride=stride, padding=kernel // 2, groups=mid, bias=False),
            nn.BatchNorm2d(mid), HSwish()
        ]
        if use_se: layers.append(SqueezeExcite(mid))
        layers += [nn.Conv2d(mid, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch)]
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x) + x if self.use_skip else self.block(x)

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.mlp = nn.Sequential(nn.Linear(channels, mid, bias=False), nn.ReLU(inplace=True), nn.Linear(mid, channels, bias=False))
    def forward(self, x):
        gate = torch.sigmoid(self.mlp(x.mean(dim=(2,3))) + self.mlp(x.amax(dim=(2,3))))
        return x * gate.view(x.size(0), x.size(1), 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
    def forward(self, x):
        gate = torch.sigmoid(self.conv(torch.cat([x.mean(dim=1,keepdim=True), x.amax(dim=1,keepdim=True)], dim=1)))
        return x * gate

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention()
    def forward(self, x): return self.sa(self.ca(x))

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU()
        )
        self.cbam = CBAM(out_ch)
        self.drop = nn.Dropout2d(0.1)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.drop(self.cbam(self.conv(torch.cat([x, skip], dim=1))))

class BoundaryRefinementHead(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(in_ch + 1, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(), nn.Conv2d(32, 1, 1))
    def forward(self, features, main_logit): return self.conv(torch.cat([features, main_logit], dim=1))

class BgRemovalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(16), HSwish())
        self.e1 = InvertedResidual(16, 16, stride=2, expand_ratio=1, kernel=3)
        self.e2 = InvertedResidual(16, 24, stride=2, expand_ratio=4.5, kernel=3)
        self.e3 = InvertedResidual(24, 40, stride=2, expand_ratio=4, kernel=5, use_se=True)
        self.e4 = InvertedResidual(40, 96, stride=2, expand_ratio=6, kernel=5, use_se=True)
        self.e5 = nn.Sequential(nn.Conv2d(96, 576, 1, bias=False), nn.BatchNorm2d(576), HSwish())
        
        self.d1 = DecoderBlock(576, 40, 192)
        self.d2 = DecoderBlock(192, 24, 96)
        self.d3 = DecoderBlock(96, 16, 48)
        self.d4 = DecoderBlock(48, 16, 24)
        
        self.d5 = nn.Sequential(nn.Conv2d(24, 8, 3, padding=1, bias=False), nn.BatchNorm2d(8), nn.GELU())
        self.out_head = nn.Conv2d(8, 1, 1)
        self.boundary_head = BoundaryRefinementHead(8)
        self.s2_head, self.s3_head, self.s4_head = nn.Conv2d(96, 1, 1), nn.Conv2d(48, 1, 1), nn.Conv2d(24, 1, 1)

    def forward(self, x):
        s = self.stem(x)
        e1 = self.e1(s)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)
        e5 = self.e5(e4)
        
        d1 = self.d1(e5, e3)
        d2 = self.d2(d1, e2)
        d3 = self.d3(d2, e1)
        d4 = self.d4(d3, s)
        
        d5_feat = self.d5(F.interpolate(d4, size=x.shape[2:], mode='bilinear', align_corners=False))
        main = self.out_head(d5_feat)

        if self.training:
            def up(t): return F.interpolate(t, size=x.shape[2:], mode='bilinear', align_corners=False) if t.shape[2:]!=x.shape[2:] else t
            return {'main': main, 'boundary': self.boundary_head(d5_feat, main), 'deep': [up(self.s2_head(d2)), up(self.s3_head(d3)), up(self.s4_head(d4))]}
        return main

class ExportWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):
        out = self.m(x)
        return torch.sigmoid(out['main'] if isinstance(out, dict) else out)

def get_pred(out): return out['main'] if isinstance(out, dict) else out
def compute_iou(prob, target):
    p, t = (prob > 0.5).float(), (target > 0.5).float()
    inter = (p * t).sum()
    return ((inter + 1e-7) / (p.sum() + t.sum() - inter + 1e-7)).item()
def tta_predict(model, img):
    model.eval()
    with torch.no_grad():
        p1 = torch.sigmoid(get_pred(model(img)))
        p2 = torch.sigmoid(get_pred(model(torch.flip(img, [3])))).flip([3])
        p3 = torch.sigmoid(get_pred(model(torch.flip(img, [2])))).flip([2])
    return (p1 + p2 + p3) / 3.0

FINAL_SIZE = 320
def build_test_loader():
    test_tf = A.Compose([A.Resize(FINAL_SIZE, FINAL_SIZE), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()])
    input_dir = '/kaggle/input'
    img_dir, msk_dir = None, None
    for root, d, f in os.walk(input_dir):
        if 'image' in root.lower() and not img_dir: img_dir = root
        if ('mask' in root.lower() or 'alpha' in root.lower()) and not msk_dir: msk_dir = root
    if not img_dir or not msk_dir: return None
    
    IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp'}
    ifiles = {Path(f).stem: os.path.join(img_dir, f) for f in os.listdir(img_dir) if Path(f).suffix.lower() in IMG_EXT}
    mfiles = {Path(f).stem: os.path.join(msk_dir, f) for f in os.listdir(msk_dir) if Path(f).suffix.lower() in IMG_EXT}
    common = sorted(set(ifiles) & set(mfiles))
    all_pairs = [(ifiles[s], mfiles[s]) for s in common]
    if not all_pairs: return None
    
    import random
    random.Random(42).shuffle(all_pairs)
    test_pairs = all_pairs[int(0.9*len(all_pairs)):]

    class TestDS(Dataset):
        def __len__(self): return len(test_pairs)
        def __getitem__(self, idx):
            i, m = test_pairs[idx]
            img = cv2.cvtColor(cv2.imread(i), cv2.COLOR_BGR2RGB)
            msk = (cv2.imread(m, cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
            aug = test_tf(image=img, mask=msk)
            return {'image': aug['image'], 'mask': aug['mask'].unsqueeze(0)}

    return DataLoader(TestDS(), batch_size=4, shuffle=False, num_workers=0, pin_memory=False)

print('Loading best checkpoint...')
model = BgRemovalNet().to(device)
if os.path.exists(f'{CHECKPOINT_DIR}/best_model.pth'):
    ckpt = torch.load(f'{CHECKPOINT_DIR}/best_model.pth', map_location=device)
    model.load_state_dict(ckpt['model_state'])
model.eval()

test_loader = build_test_loader()
if test_loader:
    print('\nEvaluating on test set with TTA...')
    t_iou, n = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            imgs, masks = batch['image'].to(device), batch['mask'].to(device)
            prob = tta_predict(model, imgs)
            t_iou += compute_iou(prob, masks)
            n += 1
    print(f'\n  TEST RESULTS (TTA)')
    print(f'  IoU : {t_iou/n:.4f}' if n>0 else 'Eval failed')

fp32_path = f'{EXPORT_DIR}/model_fp32.onnx'
int8_path = f'{EXPORT_DIR}/model_int8.onnx'

print('\nExporting FP32 ONNX...')
export_model = ExportWrapper(model).cpu()
dummy = torch.randn(1, 3, FINAL_SIZE, FINAL_SIZE)
torch.onnx.export(
    export_model, dummy, fp32_path, opset_version=13,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch', 2: 'height', 3: 'width'}, 'output': {0: 'batch', 2: 'height', 3: 'width'}}
)
model_onnx = onnx.load(fp32_path, load_external_data=True)
onnx.save(model_onnx, fp32_path, save_as_external_data=False)
print('Merged to single file')
onnx.checker.check_model(onnx.load(fp32_path))
print(f'FP32: {os.path.getsize(fp32_path)/1024/1024:.1f}MB — VALID')

print('Quantizing to INT8...')
quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QUInt8)
print(f'INT8: {os.path.getsize(int8_path)/1024/1024:.1f}MB')

print('\nValidating models on CPU...')
sess = ort.InferenceSession(fp32_path, providers=['CPUExecutionProvider'])
out = sess.run(None, {sess.get_inputs()[0].name: dummy.numpy()})[0]
print("ONNX Inference Output Shape:", out.shape)
print("PASS ✅ Export completed successfully!")
